## Snowlake gives you MANY options for synthetic data creation, let's review a few of the most popular.

The first is our native synthetic data function.

In [ ]:
## Let's establish the session that allows us to interact with databases and 
## objects within our account and set our DB & Schema variables.
from snowflake.snowpark import Session
from snowflake.snowpark.context import get_active_session
session = get_active_session()

## Set you DB & Schema variables
database = session.get_current_database()
schema = session.get_current_schema()

In [ ]:
## Remember our Census data? Let's start with this as our source table.
share = 'U_S__ZIP_CODE_METADATA.ZIP_DEMOGRAPHICS.ZIP_CODE_METADATA'
share

In [ ]:
## Pull the table into a dataframe to review
df_orig = session.table(share)
df_orig

Next, we execute the native function GENERATE_SYNTHETIC_DATA().  Notice how simple it is, all you need is your input_table & output_table.

In [ ]:
CALL SNOWFLAKE.DATA_PRIVACY.GENERATE_SYNTHETIC_DATA({
  'datasets':[
      {
        'input_table': '{{share}}',
        'output_table': '{{database}}.{{schema}}.synth_census'
      }
    ],
    'replace_output_tables':True
});

In [ ]:
df_synth = session.table('SYNTH_CENSUS')
df_synth

A non-categorical string with many unique values will be Redacted in the output unless you specify an output format with the replace option in GENERATE_SYNTHETIC_DATA.

If more than half of the values in a STRING column are unique values, Snowflake replaces the value with a redacted value in the output table due to privacy concerns.

Available Replace Values:
| Value        | Description                                                        | Example                                          |
|--------------|--------------------------------------------------------------------|--------------------------------------------------|
| `uuid`       | A UUID.                                                            | `88d99a35-c4be-4022-b06a-41fb4629b46d`         |
| `name`       | A first and last name in US locale style.                         | `George Washington`                              |
| `first_name` | A first name in US locale style.                                   | `George`                                         |
| `last_name`  | A last name in US locale style.                                    | `Washington`                                     |
| `address`    | An abbreviated address in US locale style.                         | `1600 Pennsylvania Ave`                          |
| `full_address`| A detailed street address in US locale style.                     | `1600 Pennsylvania Ave NW, Washington DC 20500` |
| `email`      | An email address.                                                  | `bdbQ6OPBS5ScOdJx8bVpFw@example.com`             |
| `phone`      | A US-style 10-digit phone number in US locale style.              | `212-555-1234`                                   |
| `ssn`        | A US-style Social Security number.                                 | `123-45-6789`                                    |

In [ ]:
df_orig.describe();

In [ ]:
df_synth.describe();

In [ ]:
age_by_state_df = df_orig.group_by('STATE').avg('median_age')
age_by_state_df_s = df_synth.group_by('STATE').avg('median_age')
joined_df = age_by_state_df.join(age_by_state_df_s, age_by_state_df['STATE'] == age_by_state_df_s['STATE'], "left")
joined_df

In [ ]:
CALL SNOWFLAKE.DATA_PRIVACY.GENERATE_SYNTHETIC_DATA({
  'datasets':[
      {
        'input_table': '{{share}}',
        'output_table': '{{database}}.{{schema}}.synth_census',
        'columns' : {
          'ZIP': {'categorical': True},
          'LATITUDE': {'categorical': True},
          'LONGITUDE': {'categorical': True},
          'CITY': {'categorical': True},
          'GEOPOINT': {'categorical': True}
          }
      }
    ],
    'replace_output_tables':True
});

In [ ]:
df_synth = session.table('SYNTH_CENSUS')
df_synth

In [ ]:
df_orig.describe();

In [ ]:
df_synth.describe();

## Now let's do an example JOIN

First, let's create two views off of the Snowflake Sample Database that has existing keys

In [ ]:
CREATE OR REPLACE VIEW syndata_db.sch.TPC_ORDERS_5K as (
    SELECT * from SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
    LIMIT 5000
);

CREATE OR REPLACE VIEW syndata_db.sch.TPC_CUSTOMERS_5K as (
    SELECT * from SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER
    LIMIT 5000
);

In [ ]:
CREATE OR REPLACE SECRET synth_secret
TYPE = SYMMETRIC_KEY
ALGORITHM = GENERIC

In [ ]:
CALL SNOWFLAKE.DATA_PRIVACY.GENERATE_SYNTHETIC_DATA({
    'datasets':[
        {
          'input_table': 'syndata_db.sch.TPC_ORDERS_5K',
          'output_table': 'syndata_db.sch.TPC_ORDERS_5K_SYNTHETIC_2',
          'columns': {'O_CUSTKEY': {'join_key': True}}
        },
        {
          'input_table': 'syndata_db.sch.TPC_CUSTOMERS_5K',
          'output_table': 'syndata_db.sch.TPC_CUSTOMERS_5K_SYNTHETIC_2',
          'columns' : {
          'C_CUSTKEY': {'join_key': True},
          'C_ADDRESS': {'categorical': True},
          'C_NAME': {'categorical': True}
          }

        }
      ],
    'consistency_secret': SYSTEM$REFERENCE('SECRET', 'synth_secret', 'SESSION', 'READ')::STRING,
      'replace_output_tables':True
  });

In [ ]:
SELECT O.O_CUSTKEY FROM syndata_db.sch.TPC_ORDERS_5K_SYNTHETIC as O
JOIN syndata_db.sch.TPC_CUSTOMERS_5K_SYNTHETIC as C
ON C.C_CUSTKEY = O.O_CUSTKEY;

In [ ]:
SELECT * FROM syndata_db.sch.TPC_ORDERS_5K_SYNTHETIC WHERE O_CUSTKEY = 781;

In [ ]:
SELECT * FROM syndata_db.sch.TPC_CUSTOMERS_5K_SYNTHETIC_2
WHERE C_CUSTKEY = 781;

In [ ]:
SELECT * FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER
WHERE C_CUSTKEY = 781;

## To wrap up native Synthetic Data creation options, you can also use SQL!

In [ ]:
SELECT CONCAT('SNOW-',UNIFORM(1000,9999, RANDOM())) AS PRODUCT_ID, 
       ABS(NORMAL(5, 3, RANDOM())) AS RATING, 
       ABS(NORMAL(750, 200::FLOAT, RANDOM())) AS PRICE
FROM TABLE(GENERATOR(ROWCOUNT => 100));

## Now let's move to more advanced options. First, because we support native Python, we can use faker!

Let's add Faker to the package selector to bring it into the session.

In [ ]:
from faker import Faker

# --- Initialize faker
fake = Faker()

# --- Number of Fake Records to Generate ---
NUM_RECORDS = 100

def generate_fake_claim():
    return {
        "NAME": fake.name(),
        "ADDRESS": fake.address(),
        "PROPERTY_VALUE": fake.random_int(min=100000, max=1000000),
        "CLAIM_AMOUNT": fake.random_int(min=1000, max=200000),
        "DISASTER_TYPE": fake.random_element(elements=("Earthquake", "Wildfire", "Flood", "Hurricane", "Tornado")),
        "DAMAGE_DESCRIPTION": fake.text(max_nb_chars=200),
        "POLICY_NUMBER": fake.bothify(text='POL-########'),
        "CLAIM_NUMBER": fake.bothify(text='CLM-########'),
        "INCIDENT_DATE": fake.date_between(start_date='-1y', end_date='today'),
        "REPORT_DATE": fake.date_between(start_date='-1y', end_date='today'),
        "CLAIM_STATUS": fake.random_element(elements=("Open", "Closed", "Pending", "Denied")),
        "DEDUCTIBLE": fake.random_int(min=500, max=5000),
        "POLICY_TYPE": fake.random_element(elements=("Homeowners", "Renters", "Commercial"))
    }

fake_claims_data = [generate_fake_claim() for _ in range(NUM_RECORDS)]
fake_claims_data

In [ ]:
snowpark_df = session.create_dataframe(fake_claims_data)
snowpark_df.write.mode("overwrite").save_as_table("faker_claim_table")

In [ ]:
SELECT * FROM faker_claim_table;

## Lastly, you can leverage AI and use Snowflake's LLM Functions to create synthetic data as well.

Let's start by creating some categories, and a table to hold them.

In [ ]:
create or replace table claim_category (
  category string
);

INSERT INTO claim_category (category) VALUES 
  ('Property'), 
  ('Automobile'), 
  ('Health'), 
  ('Business Liability'), 
  ('Disability'),
  ('Mortgage'),
  ('First-Party');

In [ ]:
create or replace table claims as (
    SELECT 
      category, 
      TRY_PARSE_JSON(
        SNOWFLAKE.CORTEX.COMPLETE(
          'llama3.1-405b',
          CONCAT(
            'Please provide 25 examples of customer claims in an insurance company for the following category:', category, '. Provide detailed and realistic scenarios that customer claim service representatives might encounter. Ensure the examples are diverse and cover various situations within each category. Please put the  examples into a JSON list. Each element in JSON list should include the following: {"scenario": <scenario>, "request": <detailed request from the customer, which usually is less than 3 sentences.>}. Only include JSON in output and no other words.'))) as tickets
    from claim_category
);

In [ ]:
SELECT * FROM CLAIMS;

In [ ]:
create or replace table flatten_claims as (
select 
    category, 
    abs(hash(value:request)) % 10000000 as id,
    value:request as request, 
    value:scenario as scenario
from claims, lateral flatten(input => tickets) 
);

In [ ]:
SELECT * FROM flatten_claims;

## We can even use the LLM to validate the data set is high quality!

In a real world scenario, this could also be used to filter out erroneous claims automatically.

In [ ]:
create or replace table rate_claims as (
    SELECT category, id, request, scenario, TRY_PARSE_JSON(SNOWFLAKE.CORTEX.COMPLETE('llama3.1-405b', CONCAT('You are a judge to verify if a claim received by an insurance company is realistic, and valid, please give scores from 1 to 5 for each category and give your final recommendation for the given question. Support Ticket: ', request, ' Please give the score in JSON format alone following this example: "{"realistic": 5, "valid": 4}".  You can put a reason into the result JSON as "reason": <reason>. Only include JSON in the output and no other words.'))) as rating
    from flatten_claims
);

In [ ]:
SELECT * FROM rate_claims;

In [ ]:
create or replace table filtered_claims as (
    select * from rate_claims where rating['realistic'] >= 4 and rating['valid'] >= 4
);

In [ ]:
SELECT * FROM filtered_claims;

## Lastly, you can also use our Native Applications.

Let's take a look at the offering from DataMynd in our Marketplace.